In [10]:
import math
import numpy as np

for i in range(5):
    print(np.exp(i/2.2))

1.0
1.5754571033903182
2.482065084623012
3.9103870686464153
6.160647084304639


In [1]:
import pandas as pd

df=pd.DataFrame()
if df.columns.tolist():
    print('Exisst')

In [20]:
import yaml
with open('../data_schema/schema.yaml', 'rb') as f:
    schema=yaml.safe_load(f)
print(schema)
import pandas as pd
df = pd.read_csv('../artifacts/20_02_26_13_57_19/ingestion/split_data/train.csv')
print(len(schema['columns'])==len(df.columns.tolist()))
validated_all_columns=True
train_column_list=df.columns.tolist()
schema_column_list=[list(col.keys())[0] for col in schema['columns']]
print(train_column_list)
print(schema_column_list)
for col in schema_column_list:
    if col not in train_column_list:
        validated_all_columns=False
print(validated_all_columns)

{'columns': [{'site_name': 'str'}, {'log_date': 'date'}, {'cellid': 'int64'}, {'ueid': 'int64'}, {'uptime': 'int64'}, {'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2_ack': 'int64'}, {'rv2_nack': 'int64'}, {'rv2_dtx': 'int64'}, {'rv2_bler': 'float64'}, {'rv3_tx': 'int64'}, {'rv3_ack': 'int64'}, {'rv3_nack': 'int64'}, {'rv3_dtx': 'int64'}, {'rv3_bler': 'float64'}, {'rca_label': 'str'}], 'numerical_columns': [{'cellid': 'int64'}, {'ueid': 'int64'}, {'uptime': 'int64'}, {'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2

In [2]:
validated_num_of_columns=True
validate_all_columns=True
validated_num_of_columns and validate_all_columns

True

In [4]:
from scipy.stats import ks_2samp
import pandas as pd
import numpy as np
columns=['cqi', 'mcs', 'ri', 'ibler', 'rbler', 'resbler', 'tbler']
train_df = pd.read_csv('../artifacts/20_02_26_11_07_56/ingestion/split_data/train.csv')
test_df = pd.read_csv('../artifacts/20_02_26_11_07_56/ingestion/split_data/test.csv')
report = {}
for column in columns:
    report[column] = np.round(ks_2samp(train_df[column], test_df[column]).pvalue, 2)
print(report)

{'cqi': np.float64(0.61), 'mcs': np.float64(0.39), 'ri': np.float64(0.75), 'ibler': np.float64(0.8), 'rbler': np.float64(0.8), 'resbler': np.float64(0.9), 'tbler': np.float64(0.8)}


In [2]:
from databricks.sql import connect
from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import yaml
load_dotenv()
query="""
SELECT *
FROM `du_stats`.`silver`.`synth_histo_table`
WHERE log_date=DATE('2026-01-01') AND ueid=17017
"""
with open('../data_schema/schema.yaml', 'r') as f:
    schema = yaml.safe_load(f)
print(schema)
columns = [list(col.keys())[0] for col in schema['columns']]
print(columns)
with connect(
    server_hostname=os.getenv('DATABRICKS_SERVER_HOSTNAME'),
    http_path=os.getenv('DATABRICKS_HTTP_PATH'),
    access_token=os.getenv('DATABRICKS_ACCESS_TOKEN')
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result=cursor.fetchall()
df = pd.DataFrame(data=result, columns=columns)
print(df.head())
import yaml
with open('../data_schema/schema.yaml','r') as f:
    numerical_schema = yaml.safe_load(f)
numerical_columns = [list(col.keys())[0] for col in numerical_schema['numerical_columns']]
df[numerical_columns].head()
arr = np.array(df)
print(arr[:5,:])

{'columns': [{'site_name': 'str'}, {'log_date': 'date'}, {'cellid': 'int64'}, {'ueid': 'int64'}, {'uptime': 'int64'}, {'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2_ack': 'int64'}, {'rv2_nack': 'int64'}, {'rv2_dtx': 'int64'}, {'rv2_bler': 'float64'}, {'rv3_tx': 'int64'}, {'rv3_ack': 'int64'}, {'rv3_nack': 'int64'}, {'rv3_dtx': 'int64'}, {'rv3_bler': 'float64'}, {'rca_label': 'str'}], 'numerical_columns': [{'ibler': 'float64'}, {'rbler': 'float64'}, {'resbler': 'float64'}, {'tbler': 'float64'}, {'cqi': 'float64'}, {'mcs': 'float64'}, {'ri': 'float64'}, {'rv0_tx': 'int64'}, {'rv0_ack': 'int64'}, {'rv0_nack': 'int64'}, {'rv0_dtx': 'int64'}, {'rv0_bler': 'float64'}, {'rv2_tx': 'int64'}, {'rv2_ack': 'int64'}, {'rv2_nack': 'int64'}, {'rv2_dtx': 'int64'},

In [4]:
import numpy as np
import pickle
with open('../artifacts/21_02_26_12_39_18/model_trainer/model/model.pkl', 'rb') as f:
    model=pickle.load(f)
np.round(model.feature_importances_, 2)

array([0.  , 0.12, 0.1 , 0.45, 0.  , 0.28, 0.03, 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])

In [2]:
from databricks.sql import connect
from dotenv import load_dotenv
import pandas as pd
import os
import yaml
load_dotenv()
schema_path = '../data_schema/schema.yaml'
with open(schema_path, 'r') as f:
    schema = yaml.safe_load(f)
columns = [list(col.keys())[0] for col in schema['columns']]
num_columns = [list(col.keys())[0] for col in schema['numerical_columns']]
query = """
SELECT * FROM du_stats.silver.synth_histo_table ORDER BY site_name, log_date, ueid
"""
with connect(
    server_hostname=os.getenv('DATABRICKS_SERVER_HOSTNAME'),
    http_path=os.getenv('DATABRICKS_HTTP_PATH'),
    access_token=os.getenv('DATABRICKS_ACCESS_TOKEN')
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
df = pd.DataFrame(data=result, columns=columns)
features_df = df[num_columns]
target_df = df.groupby(['site_name', 'log_date', 'ueid', 'cellid']).head(1)['rca_label']

In [4]:
features_df

,ibler,rbler,resbler,tbler,cqi,mcs,ri,rv0_tx,rv0_ack,rv0_nack,...,rv2_tx,rv2_ack,rv2_nack,rv2_dtx,rv2_bler,rv3_tx,rv3_ack,rv3_nack,rv3_dtx,rv3_bler
0,1.32,0.79,0.53,2.64,10.15,0.06,3.85,95,94,0,...,1,1,0,0,0.0,0,0,0,0,0.0
1,1.18,0.71,0.47,2.36,14.49,22.78,3.50,86,85,1,...,1,1,0,0,0.0,0,0,0,0,0.0
2,0.36,0.21,0.14,0.71,9.16,15.29,3.44,69,69,0,...,0,0,0,0,0.0,0,0,0,0,0.0
3,1.05,0.63,0.42,2.09,9.04,16.73,3.87,87,87,0,...,0,0,0,0,0.0,0,0,0,0,0.0
4,1.19,0.71,0.47,2.37,14.96,26.47,3.93,99,98,0,...,1,1,0,0,0.0,0,0,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2859931,1.07,0.64,0.43,2.13,13.60,0.53,3.68,72,72,0,...,0,0,0,0,0.0,0,0,0,0,0.0
2859932,2.08,1.25,0.83,4.16,13.61,7.67,3.89,76,75,1,...,1,1,0,0,0.0,0,0,0,0,0.0
2859933,1.72,1.03,0.69,3.44,6.41,0.17,3.71,52,52,0,...,0,0,0,0,0.0,0,0,0,0,0.0
2859934,1.74,1.04,0.69,3.47,12.97,2.89,3.61,94,93,0,...,1,1,0,0,0.0,0,0,0,0,0.0


In [5]:
target_df

0                            GOOD
1                            GOOD
2                            GOOD
2883                  BAD CHANNEL
2884                  BAD CHANNEL
                    ...          
2854171    GOOD CHANNEL HIGH BLER
2854172    GOOD CHANNEL HIGH BLER
2857053         SCHEDULER LIMITED
2857054         SCHEDULER LIMITED
2857055         SCHEDULER LIMITED
Name: rca_label, Length: 2976, dtype: object

In [8]:
import torch
tensor = torch.Tensor(features_df.values)
print(tensor.shape)
tensor_reshaped = tensor.reshape((-1, 961, 22))
print(tensor_reshaped.shape)
single_session = tensor_reshaped[0]
print(single_session.shape)
print(single_session[0])

torch.Size([2859936, 22])
torch.Size([2976, 961, 22])
torch.Size([961, 22])
tensor([1.3200e+00, 7.9000e-01, 5.3000e-01, 2.6400e+00, 1.0150e+01, 6.0000e-02,
        3.8500e+00, 9.5000e+01, 9.4000e+01, 0.0000e+00, 1.0000e+00, 1.0500e+00,
        1.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00])


In [11]:
import torch
X, y = torch.load('../artifacts/03_03_26_19_37_25/transformation/transformed_data/train.pt')
print(X.shape)
print(y.shape)

torch.Size([844, 961, 22])
torch.Size([844])
